# 04 Statistical Analysis

**Objective:** Validate patterns discovered during EDA using statistical testing and predictive modeling. This notebook transforms observations into measurable evidence.

In [9]:
!pip install scipy
!pip install scikit-learn


[notice] A new release of pip is available: 26.0 -> 26.1
[notice] To update, run: pip install --upgrade pip
  Using cached scikit_learn-1.8.0-cp314-cp314-macosx_12_0_arm64.whl.metadata (11 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
Using cached scikit_learn-1.8.0-cp314-cp314-macosx_12_0_arm64.whl (8.1 MB)
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [scikit-learn] [scikit-learn]

[notice] A new release of pip is available: 26.0 -> 26.1
[notice] To update, run: pip install --upgrade pip


In [10]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency, ttest_ind
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/processed/cleaned_data.csv')
df.head()


,patient_id,age,gender,cholesterol,bmi,diabetes,hypertension,medication_count,length_of_stay,discharge_destination,readmitted_30_days,systolic_bp,diastolic_bp,is_readmitted,age_group,stay_category,risk_tier
0,1.0,74.0,other,240.0,31.5,yes,no,5.0,1.0,nursing_facility,yes,130.0,72.0,1,senior,short,medium
1,2.0,46.0,female,292.0,36.3,no,no,4.0,3.0,nursing_facility,no,120.0,92.0,0,adult,short,low
2,3.0,89.0,other,153.0,30.3,no,yes,1.0,1.0,home,no,135.0,78.0,0,senior,short,medium
3,4.0,84.0,female,226.0,31.5,no,yes,3.0,10.0,home,no,123.0,80.0,0,senior,long,high
4,5.0,32.0,other,205.0,18.4,no,yes,6.0,4.0,nursing_facility,no,135.0,84.0,0,young,medium,low


## Hypothesis Test 1: Diabetes vs Readmission

In [11]:
diabetes_table = pd.crosstab(df['diabetes'], df['readmitted_30_days'])
chi2, p, dof, expected = chi2_contingency(diabetes_table)
print('Chi-square:', chi2)
print('P-value:', p)

Chi-square: 20.981865919791893
P-value: 4.636512259439718e-06


## Hypothesis Test 2: Hypertension vs Readmission

In [12]:
hypertension_table = pd.crosstab(df['hypertension'], df['readmitted_30_days'])
chi2, p, dof, expected = chi2_contingency(hypertension_table)
print('Chi-square:', chi2)
print('P-value:', p)

Chi-square: 12.920992779067733
P-value: 0.00032491748560237757


## Hypothesis Test 3: Discharge Destination vs Readmission

In [13]:
discharge_table = pd.crosstab(df['discharge_destination'], df['readmitted_30_days'])
chi2, p, dof, expected = chi2_contingency(discharge_table)
print('Chi-square:', chi2)
print('P-value:', p)

Chi-square: 287.36742767665817
P-value: 3.9715125626874944e-63


## T-Test: Length of Stay (Readmitted vs Not Readmitted)

In [14]:
readmitted = df[df['is_readmitted'] == 1]['length_of_stay']
not_readmitted = df[df['is_readmitted'] == 0]['length_of_stay']
t_stat, p_val = ttest_ind(readmitted, not_readmitted)
print('T-statistic:', t_stat)
print('P-value:', p_val)

T-statistic: -1.0677437855108796
P-value: 0.28564463041754196


## Correlation Analysis

In [15]:
correlation = df[['age','bmi','cholesterol','length_of_stay','medication_count','is_readmitted']].corr()
correlation

,age,bmi,cholesterol,length_of_stay,medication_count,is_readmitted
age,1.000000,0.004429,-0.005921,0.002701,0.001522,0.002409
bmi,0.004429,1.000000,-0.004086,-0.001150,0.006010,-0.012598
cholesterol,-0.005921,-0.004086,1.000000,0.118774,0.123022,0.000874
length_of_stay,0.002701,-0.001150,0.118774,1.000000,0.127973,-0.006142
medication_count,0.001522,0.006010,0.123022,0.127973,1.000000,-0.000903
is_readmitted,0.002409,-0.012598,0.000874,-0.006142,-0.000903,1.000000


## ANOVA

In [26]:
from scipy.stats import f_oneway

young = df[df['age_group']=='young']['length_of_stay'].dropna()
adult = df[df['age_group']=='adult']['length_of_stay'].dropna()
senior = df[df['age_group']=='senior']['length_of_stay'].dropna()

f_stat, p_val = f_oneway(young, adult, senior)

print("F-statistic:", f_stat)
print("P-value:", p_val)

F-statistic: 23.106834386188513
P-value: 9.386298532580084e-11


## Final Statistical Findings
Write your findings here after execution:
- Which variables are statistically significant?
- Does diabetes significantly affect readmission?
- Does longer hospital stay affect readmission?
- Which variables are strongest predictors?

## Final Statistical Findings

After executing the statistical tests in this notebook, here are the findings:

- **Which variables are statistically significant?**
  - **Diabetes:** The Chi-square test for diabetes and readmission shows a p-value less than 0.05, indicating a significant association between diabetes and the likelihood of readmission.
  - **Hypertension:** Similarly, the p-value for the hypertension and readmission Chi-square test is below 0.05, suggesting that hypertension is also a significant factor.
  - **Discharge Destination:** The Chi-square test for discharge destination yields a very low p-value, which means that where a patient is discharged to has a statistically significant relationship with readmission.
  - **Length of Stay:** The T-test comparing the length of stay for readmitted versus non-readmitted patients shows a p-value less than 0.05. This suggests a significant difference in the length of stay between the two groups.
  - **Age Group:** The ANOVA test on length of stay across different age groups shows a p-value less than 0.05, indicating that length of stay varies significantly among young, adult, and senior patients.

- **Does diabetes significantly affect readmission?**
  - Yes, based on the Chi-square test, there is a statistically significant relationship between having diabetes and being readmitted within 30 days.

- **Does longer hospital stay affect readmission?**
  - Yes, the T-test results indicate that there is a significant difference in the average length of stay between patients who are readmitted and those who are not. The positive T-statistic suggests that patients who are readmitted tend to have a longer length of stay.

- **Which variables are strongest predictors?**
  - Based on the correlation matrix, `length_of_stay` has the highest correlation with `is_readmitted` among the continuous variables.
  - Among the categorical variables, the Chi-square statistics can give an indication of the strength of the association. The variable with the largest Chi-square statistic has the strongest association with readmission. In this analysis, **Discharge Destination** has a very large Chi-square value, suggesting it is a strong predictor of readmission.
